# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |RM032-Ethnic_group_by_age |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File | RM032_Ethnic_group_by_age|
| Output File |ethnic_group_by_age.json|
| Notebook Author | Peter |
| Last Updated | |

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [4]:
# File paths

input_file = "RM032_Ethnic_group_by_age.xlsx"
output_file = "ethnic_group_by_age.json"

raw_data_path = Path("../../data/demographic/raw/ethnic_group")
output_data_path = Path("../../data/demographic/processed/ethnic_group")

## 3. Load Dataset

In [5]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Ethnic group (20 categories) Code,Ethnic group (20 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0,Aged under 1 year,0
1,E06000001,Hartlepool,-8,Does not apply,1,Aged 1 year,0
2,E06000001,Hartlepool,-8,Does not apply,2,Aged 2 years,0
3,E06000001,Hartlepool,-8,Does not apply,3,Aged 3 years,0
4,E06000001,Hartlepool,-8,Does not apply,4,Aged 4 years,0


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [6]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 567600 entries, 0 to 567599
Data columns (total 7 columns):
 #   Column                             Non-Null Count   Dtype 
---  ------                             --------------   ----- 
 0   Lower tier local authorities Code  567600 non-null  object
 1   Lower tier local authorities       567600 non-null  object
 2   Ethnic group (20 categories) Code  567600 non-null  int64 
 3   Ethnic group (20 categories)       567600 non-null  object
 4   Age (86 categories) Code           567600 non-null  int64 
 5   Age (86 categories)                567600 non-null  object
 6   Observation                        567600 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 30.3+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Ethnic group (20 categories) Code', 'Ethnic group (20 categories)',
       'Age (86 categories) Code', 'Age (86 categories)', 'Observation'],
      dtype='object')

In [7]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Ethnic group (20 categories) Code,Ethnic group (20 categories),Age (86 categories) Code,Age (86 categories),Observation
count,567600,567600,567600.000000,567600,567600.000000,567600,567600.000000
unique,330,330,NaN,20,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Does not apply,NaN,Aged under 1 year,NaN
freq,1720,1720,NaN,28380,NaN,6600,NaN
mean,NaN,NaN,9.100000,NaN,42.500000,NaN,104.996505
std,NaN,NaN,6.624959,NaN,24.824405,NaN,426.484139
min,NaN,NaN,-8.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,4.750000,NaN,21.000000,NaN,1.000000
50%,NaN,NaN,9.500000,NaN,42.500000,NaN,5.000000
75%,NaN,NaN,14.250000,NaN,64.000000,NaN,23.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [8]:
# Cleaning code

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [9]:
# Dataset-specific processing
selected_columns = ['Lower tier local authorities',
                    'Ethnic group (20 categories)',
                    'Age (86 categories) Code',
                    'Observation']

df_processed = df_processed[selected_columns]
df_processed = df_processed.rename(columns={
    'Ethnic group (20 categories)': 'Ethnic Group',
    'Age (86 categories) Code': 'Age',
    'Observation': 'Ethnic Count'
})

In [10]:
from ontology.age_ontology import (
    age_group,
    age_order
)

In [11]:
def get_ethnic_group_by_age(local_authority):
    filtered_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    filtered_data['Age'] = filtered_data['Age'].apply(age_group)

    result = (filtered_data.groupby(['Lower tier local authorities', 
                                     'Ethnic Group',
                                     'Age'],as_index=False)['Ethnic Count'].sum())

    result['Age'] = pd.Categorical(result['Age'], categories=age_order, ordered=True)

    result = result.sort_values('Age').reset_index(drop=True)

    return result

In [12]:
get_ethnic_group_by_age('Manchester')

,Lower tier local authorities,Ethnic Group,Age,Ethnic Count
0,Manchester,"Asian, Asian British or Asian Welsh: Bangladeshi",0-4,833
1,Manchester,"Black, Black British, Black Welsh, Caribbean o...",0-4,3717
2,Manchester,"Asian, Asian British or Asian Welsh: Other Asian",0-4,1030
3,Manchester,White: Irish,0-4,95
4,Manchester,"Black, Black British, Black Welsh, Caribbean o...",0-4,438
...,...,...,...,...
175,Manchester,Other ethnic group: Any other ethnic group,75+,165
176,Manchester,Other ethnic group: Arab,75+,101
177,Manchester,"White: English, Welsh, Scottish, Northern Iris...",75+,15939
178,Manchester,White: Irish,75+,1676


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [14]:
get_ethnic_group_by_age('Manchester')['Ethnic Count'].sum()

np.int64(551947)

In [15]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 567600 entries, 0 to 567599
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Lower tier local authorities  567600 non-null  object
 1   Ethnic Group                  567600 non-null  object
 2   Age                           567600 non-null  int64 
 3   Ethnic Count                  567600 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 17.3+ MB


Lower tier local authorities    0
Ethnic Group                    0
Age                             0
Ethnic Count                    0
dtype: int64

## 8. Export Processed Dataset

In [16]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df_processed.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

ethinic_group_by_age = get_ethnic_group_by_age('Manchester')
ethinic_group_by_age.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported